In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re 

from cherab.openadas import OpenADAS, parse
from cherab.core.atomic import carbon

In [2]:
adas = OpenADAS(permit_extrapolation=True)

In [3]:
def new_spectroscopyline_identifier(line):
    # Checando se linha tem header
    return '/TYPE' in line and "A" in line

def creating_table_of_Pecs(datas):
    matrix_per_wl = []
    comprimentos = []
    
    idx = 0
    total_linhas = len(datas)
    
    while idx < total_linhas:
        row = datas[idx]
        
        if new_spectroscopyline_identifier(row):
            match = re.search(r"(\d+\.?\d*)A", row)
            
            if match:
                wavelength = float(match.group(1)) # Convertido para float
                comprimentos.append(wavelength)
                
                # Extrai o tipo (/TYPE = EXCIT ou RECOM)
                match_type_search = re.search(r"/TYPE\s*=\s*(\w+)", row)
                match_type = match_type_search.group(1) if match_type_search else "UNKNOWN"
                
                # Salta as linhas de metadados intermediárias (ajuste o 7 se necessário)
                idx += 7

                values = []

                # Lê as linhas da matriz
                for _ in range(int(3 * 24)):
                    if idx < total_linhas:
                        row_data = datas[idx]
                        
                        # Trata notação científica Fortran (D -> E) e separa por espaço
                        for num in row_data.replace('D', 'E').split():
                            values.append(float(num))

                        idx += 1
                
                # Redimensiona para matriz
                matrix_pec = np.array(values).reshape(24, 21)
                
                # Dicionário AGORA DENTRO do 'if match:'
                wavelength_matrix = {
                    "Wavelength": wavelength,
                    "Type": match_type,
                    "matrix_pec": matrix_pec
                }
                
                matrix_per_wl.append(wavelength_matrix)
                
                # Como o 'for' já avançou o 'idx' até o próximo bloco, 
                # usarmos 'continue' para não rodar o 'idx += 1' do final do while
                continue 
                
        # Avança o índice se a linha atual NÃO for um cabeçalho
        idx += 1
                
    return matrix_per_wl, comprimentos


def read_archive(path):
    if type(path) != str:
       return print("Please inform a valid path")
    
    with open(path,'r') as archive:
        lines = [row.strip() for row in archive]
        lines.pop(0)            
    return lines
    


In [4]:
a = read_archive("pecs96.dat")

In [5]:
lista_pecs, lista_comprimentos = creating_table_of_Pecs(a)

In [12]:
lista_pecs[1]["matrix_pec"][7,8]

9.3e-13

In [11]:
lista_comprimentos[1]

28.5

In [10]:
lista_pecs[1]["Wavelength"]

28.5